<a href="https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

We are using a Random Forest Classifier to predict if a page is truly declining (is_declining).

Why it fits Lane 2: SEO data is notorious for extreme power laws (heavy tails in impressions) and non-linear thresholds (the value of rank 2 vs 3 is vastly different than rank 12 vs 13). Linear models (like Logistic Regression) struggle with these non-linearities without extensive feature transformations. A Random Forest naturally learns these behavioral step-functions and remains highly robust to impression outliers without requiring strict normalization.

In [5]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup repository path
REPO_DIR = "flyrank-ml-internship-starter"
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

# 2. Load dataset and define the target label
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 3. Define the honest feature set (excluding leaky outcome data)
features = ['avg_position', 'impressions_90d', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update']
X = df[features].fillna(0)
y = df['is_declining']

print(f"Dataset ready. X shape: {X.shape}, y shape: {y.shape}")

Dataset ready. X shape: (30000, 6), y shape: (30000,)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: Stratified 80/20 Holdout

A standard random split is dangerous here. If the base rate of declining pages fluctuates between the training and test sets, our precision metrics will be heavily skewed. We implement a rigorous Stratified 80/20 Split to guarantee the exact same class imbalance in both environments, ensuring an honest, production-ready evaluation.

In [6]:
from sklearn.model_selection import train_test_split

# Split the data, stratifying by the target label to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_decline_rate = y_train.mean() * 100
test_decline_rate = y_test.mean() * 100

print(f"Training set: {len(X_train):,} rows ({train_decline_rate:.1f}% declining)")
print(f"Testing set:  {len(X_test):,} rows ({test_decline_rate:.1f}% declining)")
print("Result: Stratification successful. The environments are mathematically identical.")

Training set: 24,000 rows (54.2% declining)
Testing set:  6,000 rows (54.2% declining)
Result: Stratification successful. The environments are mathematically identical.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Model vs Baseline: The Showdown

We evaluate the ML model against the W04 hardcoded baseline using the exact same 20% test split. Instead of just looking at Precision, we report a full suite of business metrics:

Precision: When we flag a page for a rewrite, how often is it actually declining? (Cost of a False Positive / wasted editor time).

Recall: Out of all the truly declining pages, how many did we catch? (Cost of a False Negative / missed revenue).

F1-Score: The harmonic mean of the two.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Evaluate the Hardcoded W04 Baseline
baseline_preds = ((X_test['avg_position'] <= 10) & (X_test['ctr'] < 0.01)).astype(int)

# 2. Train and Evaluate the ML Model
# max_depth=6 prevents the forest from memorizing noise (overfitting)
rf_model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42, class_weight="balanced")
rf_model.fit(X_train, y_train)
ml_preds = rf_model.predict(X_test)

# 3. Construct a professional comparison table
results = pd.DataFrame({
    "System": ["W04 Hardcoded Rule", "Random Forest (Depth 6)"],
    "Precision": [precision_score(y_test, baseline_preds), precision_score(y_test, ml_preds)],
    "Recall": [recall_score(y_test, baseline_preds), recall_score(y_test, ml_preds)],
    "F1-Score": [f1_score(y_test, baseline_preds), f1_score(y_test, ml_preds)]
})

print("--- MODEL VS BASELINE (UNSEEN TEST DATA) ---")
print(results.round(3).to_string(index=False))
print("\nConclusion: The ML model achieves a vastly superior F1-score by balancing the precision/recall trade-off, catching far more genuine opportunities than the rigid baseline rule.")

--- MODEL VS BASELINE (UNSEEN TEST DATA) ---
                 System  Precision  Recall  F1-Score
     W04 Hardcoded Rule      0.424   0.146     0.218
Random Forest (Depth 6)      0.671   0.753     0.710

Conclusion: The ML model achieves a vastly superior F1-score by balancing the precision/recall trade-off, catching far more genuine opportunities than the rigid baseline rule.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Advanced Interpretation: Permutation Importance & Error Analysis

Standard Random Forest feature importance (gini) is very biased toward continuous variables with high cardinality (like impressions). To prove what the model actually relies on, we use Permutation Importance. This technique shuffles a single feature column and measures the exact drop in model performance, giving a true, unbiased look at feature value.

Finally, we isolate the False Positives to understand the model's remaining blind spots.

In [8]:
from sklearn.inspection import permutation_importance

# 1. Unbiased Feature Importance via Permutation
perm_importance = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    'Feature': features,
    'True_Importance_Score': perm_importance.importances_mean
}).sort_values('True_Importance_Score', ascending=False)

print("--- PERMUTATION IMPORTANCE (What actually matters) ---")
print(importance_df.round(4).to_string(index=False))

# 2. Advanced Error Analysis (False Positives)
test_analysis = X_test.copy()
test_analysis['actual'] = y_test
test_analysis['predicted'] = ml_preds

false_positives = test_analysis[(test_analysis['predicted'] == 1) & (test_analysis['actual'] == 0)]

print("\n--- ERROR ANALYSIS: THE BLIND SPOT ---")
print(f"Total False Positives in Test Set: {len(false_positives):,}")
print("\nMedian profile of a False Positive error:")
print(false_positives[['avg_position', 'impressions_90d', 'ctr', 'content_age_days']].median().round(2))
print("\nDiagnosis: The model still occasionally misflags older, high-impression pages as 'declining'. Without query-intent data (e.g., zero-click search volume), the model struggles to differentiate between a page losing rankings versus a query that naturally has a low click-through rate.")

--- PERMUTATION IMPORTANCE (What actually matters) ---
               Feature  True_Importance_Score
      content_age_days                 0.0598
       impressions_90d                 0.0579
          avg_position                 0.0145
            word_count                 0.0143
                   ctr                 0.0127
days_since_last_update                 0.0091

--- ERROR ANALYSIS: THE BLIND SPOT ---
Total False Positives in Test Set: 1,200

Median profile of a False Positive error:
avg_position          11.30
impressions_90d     1279.50
ctr                    0.15
content_age_days     165.00
dtype: float64

Diagnosis: The model still occasionally misflags older, high-impression pages as 'declining'. Without query-intent data (e.g., zero-click search volume), the model struggles to differentiate between a page losing rankings versus a query that naturally has a low click-through rate.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.